# 04 - Structural Filtering, Chunk-Level (ContraDoc)

KG-pattern queries that surface candidate contradiction pairs. The patterns still operate on the `:Entity`-`:RELATION`-`:Entity` triple subgraph, but we aggregate matched triple pairs to **unique chunk pairs** via their `sentence_id`s so output is directly comparable with step 05's vector retrieval.

## Patterns (S-SR / S-SO / S-Union)

- **S-SR** — same subject + same predicate, different object. Catches numeric / factual swaps.
- **S-SO** — same subject + same object, different predicate. Catches antonym / negation / relation switches.
- **S-Union** — S-SR ∪ S-SO.

Both queries exclude same-sentence matches (`r1.sentence_id <> r2.sentence_id`) since the pipeline retrieves chunk pairs, not intra-chunk triple pairs.

**Output:** `data/processed/ContraDoc/structural_candidates.jsonl` — one JSON object per candidate chunk pair.

In [1]:
import json
from collections import defaultdict
from pathlib import Path

from neo4j import GraphDatabase

from config import settings

OUTPUT_PATH = Path("data/processed/ContraDoc/structural_candidates.jsonl")

## Connect to Neo4j and pull chunks / gold pairs

In [2]:
driver = GraphDatabase.driver(
    settings.neo4j_uri,
    auth=(settings.neo4j_user, settings.neo4j_password.get_secret_value()),
)
driver.verify_connectivity()


def run(cypher, **params):
    with driver.session() as s:
        return list(s.run(cypher, **params))


n_docs = run("MATCH (d:Document) RETURN count(d) AS n")[0]["n"]
n_chunks = run("MATCH (c:Chunk) RETURN count(c) AS n")[0]["n"]
n_rels = run("MATCH ()-[r:RELATION]->() RETURN count(r) AS n")[0]["n"]
print(f"Connected. {n_docs} documents, {n_chunks} chunks, {n_rels} :RELATION edges.")

chunks = [
    dict(r)
    for r in run(
        "MATCH (c:Chunk) RETURN c.doc_id AS doc_id, c.sentence_id AS sid, c.source_text AS text, "
        "c.is_gold_evidence AS is_gold_evidence, c.is_gold_ref AS is_gold_ref, c.ref_index AS ref_index"
    )
]
chunk_by_key = {(c["doc_id"], c["sid"]): c for c in chunks}
doc_label = {r["d"]: r["l"] for r in run("MATCH (d:Document) RETURN d.doc_id AS d, d.contradiction AS l")}


def pkey(a, b):
    return (a, b) if a < b else (b, a)


gold_pairs = set()
for ev in chunks:
    if not ev["is_gold_evidence"]:
        continue
    for ref in chunks:
        if ref["is_gold_ref"] and ref["doc_id"] == ev["doc_id"]:
            gold_pairs.add(pkey((ev["doc_id"], ev["sid"]), (ref["doc_id"], ref["sid"])))

n_gold_usable_docs = len({p[0][0] for p in gold_pairs})
print(f"Gold chunk pairs: {len(gold_pairs)} across {n_gold_usable_docs} docs")

Connected. 100 documents, 3664 chunks, 7129 :RELATION edges.
Gold chunk pairs: 37 across 36 docs


## S-SR / S-SO queries returning DISTINCT chunk pairs

In [3]:
S_SR = """
MATCH (s:Entity)-[r1:RELATION]->(o1:Entity), (s)-[r2:RELATION]->(o2:Entity)
WHERE r1.doc_id = r2.doc_id
  AND r1.predicate = r2.predicate
  AND elementId(r1) < elementId(r2)
  AND o1.name <> o2.name
  AND r1.sentence_id <> r2.sentence_id
RETURN DISTINCT r1.doc_id AS doc_id, r1.sentence_id AS sid_a, r2.sentence_id AS sid_b
"""

S_SO = """
MATCH (s:Entity)-[r1:RELATION]->(o:Entity), (s)-[r2:RELATION]->(o)
WHERE r1.doc_id = r2.doc_id
  AND r1.predicate <> r2.predicate
  AND elementId(r1) < elementId(r2)
  AND r1.sentence_id <> r2.sentence_id
RETURN DISTINCT r1.doc_id AS doc_id, r1.sentence_id AS sid_a, r2.sentence_id AS sid_b
"""


def rows_to_pairs(rows, pattern):
    out = []
    for r in rows:
        key = pkey((r["doc_id"], r["sid_a"]), (r["doc_id"], r["sid_b"]))
        out.append({"doc_id": r["doc_id"], "pattern": pattern, "sid_a": key[0][1], "sid_b": key[1][1]})
    # Dedupe
    seen = set()
    deduped = []
    for o in out:
        k = (o["doc_id"], o["sid_a"], o["sid_b"])
        if k not in seen:
            seen.add(k)
            deduped.append(o)
    return deduped


sr_records = rows_to_pairs(run(S_SR), "S-SR")
so_records = rows_to_pairs(run(S_SO), "S-SO")

sr_pairs = {((r["doc_id"], r["sid_a"]), (r["doc_id"], r["sid_b"])) for r in sr_records}
so_pairs = {((r["doc_id"], r["sid_a"]), (r["doc_id"], r["sid_b"])) for r in so_records}
union_pairs = sr_pairs | so_pairs

print(f"S-SR  chunk pairs: {len(sr_pairs)}")
print(f"S-SO  chunk pairs: {len(so_pairs)}")
print(f"S-Union chunk pairs: {len(union_pairs)}")

S-SR  chunk pairs: 459
S-SO  chunk pairs: 109
S-Union chunk pairs: 565


## Evaluation

In [4]:
def evaluate(pairs: set, name: str) -> dict:
    caught = pairs & gold_pairs
    docs_caught = {p[0][0] for p in caught}
    per_doc = defaultdict(int)
    for a, _ in pairs:
        per_doc[a[0]] += 1
    yes_vol = [per_doc.get(d, 0) for d, l in doc_label.items() if l == "YES"]
    no_vol = [per_doc.get(d, 0) for d, l in doc_label.items() if l == "NO"]
    return {
        "name": name,
        "n": len(pairs),
        "caught": len(caught),
        "pair_r": len(caught) / max(len(gold_pairs), 1),
        "doc_r": len(docs_caught) / max(n_gold_usable_docs, 1),
        "docs_caught": len(docs_caught),
        "prec": len(caught) / max(len(pairs), 1),
        "yes_mean": sum(yes_vol) / max(len(yes_vol), 1),
        "no_mean": sum(no_vol) / max(len(no_vol), 1),
    }


rows = [evaluate(sr_pairs, "S-SR"), evaluate(so_pairs, "S-SO"), evaluate(union_pairs, "S-Union")]
print(f"Gold chunk pairs: {len(gold_pairs)} across {n_gold_usable_docs} docs")
print()
header = f"{'Method':10} {'#cand':>6}  {'caught':>6}  {'Pair-R':>7}  {'Doc-R':>11}  {'Prec':>6}  {'YES mean':>9}  {'NO mean':>9}"
print(header)
print("-" * len(header))
for r in rows:
    print(
        f"{r['name']:10} {r['n']:>6}  {r['caught']:>6}  "
        f"{r['pair_r']:>6.1%}  "
        f"{r['docs_caught']:>2}/{n_gold_usable_docs:<2} {r['doc_r']:>5.1%}  "
        f"{r['prec']:>5.1%}  "
        f"{r['yes_mean']:>8.1f}  {r['no_mean']:>8.1f}"
    )

Gold chunk pairs: 37 across 36 docs

Method      #cand  caught   Pair-R        Doc-R    Prec   YES mean    NO mean
-----------------------------------------------------------------------------
S-SR          459      12   32.4%  12/36 33.3%   2.6%       3.8       5.4
S-SO          109       4   10.8%   4/36 11.1%   3.7%       1.1       1.1
S-Union       565      16   43.2%  16/36 44.4%   2.8%       4.8       6.5


## Save S-Union chunk pairs

In [5]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for key_a, key_b in union_pairs:
        ca, cb = chunk_by_key[key_a], chunk_by_key[key_b]
        pattern = "S-SR+S-SO" if (key_a, key_b) in (sr_pairs & so_pairs) else "S-SR" if (key_a, key_b) in sr_pairs else "S-SO"
        rec = {
            "doc_id": ca["doc_id"],
            "pattern": pattern,
            "is_gold_pair": (key_a, key_b) in gold_pairs,
            "chunk_a": {
                "sentence_id": ca["sid"],
                "source_text": ca["text"],
                "is_gold_evidence": ca["is_gold_evidence"],
                "is_gold_ref": ca["is_gold_ref"],
            },
            "chunk_b": {
                "sentence_id": cb["sid"],
                "source_text": cb["text"],
                "is_gold_evidence": cb["is_gold_evidence"],
                "is_gold_ref": cb["is_gold_ref"],
            },
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Saved {len(union_pairs)} S-Union chunk pairs -> {OUTPUT_PATH.resolve()}")

caught_sr = [p for p in sr_pairs if p in gold_pairs]
caught_so = [p for p in so_pairs if p in gold_pairs]
for label, caught in [("S-SR", caught_sr), ("S-SO", caught_so)]:
    print()
    print(f"Sample caught gold pair ({label}, n={len(caught)}):")
    if caught:
        key_a, key_b = caught[0]
        ca, cb = chunk_by_key[key_a], chunk_by_key[key_b]
        tag_a = "EV" if ca["is_gold_evidence"] else ("REF" if ca["is_gold_ref"] else "  ")
        tag_b = "EV" if cb["is_gold_evidence"] else ("REF" if cb["is_gold_ref"] else "  ")
        print(f"  doc_id={ca['doc_id']}")
        print(f"  [{tag_a}] sid={ca['sid']}: {ca['text'][:140]}")
        print(f"  [{tag_b}] sid={cb['sid']}: {cb['text'][:140]}")

driver.close()

Saved 565 S-Union chunk pairs -> D:\AT82.05-Claim-Contradiction-Over-Knowledge-Graphs\experiments\data\processed\ContraDoc\structural_candidates.jsonl

Sample caught gold pair (S-SR, n=12):
  doc_id=3503017493_6
  [EV] sid=3: The Hope Highway was first established in 1950.
  [REF] sid=22: The Hope Highway was first established in 1928.

Sample caught gold pair (S-SO, n=4):
  doc_id=3488771854_10
  [REF] sid=16: Meanwhile Melinda continues to discourage Worthy, until going to a fortune teller (in fact Kite in disguise), where she is convinced to rele
  [EV] sid=23: Melinda remains unconvinced and rejects Worthy's courtship.


## Per-type recall breakdown

How well does each structural pattern recover gold pairs for each ContraDoc contradiction type? Multi-label docs (e.g., `Content|Numeric`) contribute to every listed type.

In [6]:
driver = GraphDatabase.driver(
    settings.neo4j_uri,
    auth=(settings.neo4j_user, settings.neo4j_password.get_secret_value()),
)
with driver.session() as _s:
    doc_types = {
        r["doc_id"]: [t for t in (r["contra_type"] or "none").split("|") if t]
        for r in _s.run("MATCH (d:Document {contradiction: 'YES'}) RETURN d.doc_id AS doc_id, d.contra_type AS contra_type")
    }
driver.close()


def per_type_recall(pairs, name):
    type_totals = defaultdict(int)
    type_caught = defaultdict(int)
    for p in gold_pairs:
        doc_id = p[0][0]
        for t in doc_types.get(doc_id, ["unknown"]):
            type_totals[t] += 1
            if p in pairs:
                type_caught[t] += 1
    all_types = sorted(type_totals.keys(), key=lambda x: -type_totals[x])
    print(f"\n{name}:")
    print(f"  {'type':30s}  caught  total  recall")
    print("  " + "-" * 52)
    for t in all_types:
        caught, total = type_caught[t], type_totals[t]
        print(f"  {t:30s}  {caught:>6}  {total:>5}  {caught / max(total, 1):>6.1%}")


for name, pairs in [("S-SR", sr_pairs), ("S-SO", so_pairs), ("S-Union", union_pairs)]:
    per_type_recall(pairs, name)


S-SR:
  type                            caught  total  recall
  ----------------------------------------------------
  Content                             11     23   47.8%
  Numeric                              3     10   30.0%
  Negation                             0      6    0.0%
  Perspective/View/Opinion             2      6   33.3%
  Factual                              1      5   20.0%
  Emotion/Mood/Feeling                 0      4    0.0%
  Causal                               0      2    0.0%
  Relation                             0      1    0.0%

S-SO:
  type                            caught  total  recall
  ----------------------------------------------------
  Content                              3     23   13.0%
  Numeric                              1     10   10.0%
  Negation                             2      6   33.3%
  Perspective/View/Opinion             2      6   33.3%
  Factual                              0      5    0.0%
  Emotion/Mood/Feeling              